# Embedding2Adapter for SLM RAG

Run the cells from top to bottom. Dataset preparation calls Ollama to create embeddings and can take time. The final cell starts HyperNet training.

In [1]:
from collections import Counter
from pathlib import Path
from pprint import pprint
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Project root could not be found.")

project_root = str(PROJECT_ROOT)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Experiment.src.utils.pipeline import Pipeline

## 1. Initialize the pipeline

This loads and preprocesses the configured datasets once. Models are loaded lazily when a pipeline is selected.

In [2]:
pipeline = Pipeline()

Dataset: HotPotQA Initialized!
Dataset: Musique Initialized!
Dataset: TwoWikiMultihopQA Initialized!
Dataset: MultiHopRAG Initialized!
Dataset Constructor Initialized!


## 2. Inspect and validate the datasets

In [5]:
fields = pipeline.dataset_constructor.OUTPUT_FIELDS

for dataset_name, dataset in pipeline.dataset_constructor.datasets.items():
    print(f"\n=== {dataset_name} ===")
    print(f"train rows: {len(dataset.train_ds['query']):,}")
    print(f"val rows  : {len(dataset.val_ds['query']):,}")
    print(f"eval rows : {len(dataset.eval_ds['query']):,}")

    for split_name, split in (
        ("train", dataset.train_ds),
        ("validation", dataset.val_ds),
        ("eval", dataset.eval_ds),
    ):
        field_lengths = {field: len(split[field]) for field in fields}
        if len(set(field_lengths.values())) != 1:
            raise ValueError(
                f"Misaligned fields in {dataset_name}/{split_name}: {field_lengths}"
            )

    context_counts = Counter(
        len(gold) + len(distractors)
        for gold, distractors in zip(
            dataset.train_ds["gold_context"],
            dataset.train_ds["distractor"],
        )
    )
    print("train contexts per question:", dict(sorted(context_counts.items())))

print("\nAll dataset fields are aligned.")


=== hotpotqa/hotpot_qa ===
train rows: 3,000
val rows  : 150
eval rows : 1,000
train contexts per question: {3: 3, 4: 1, 5: 2996}

=== awinml/musique ===
train rows: 2,500
val rows  : 125
eval rows : 500
train contexts per question: {5: 2500}

=== framolfese/2WikiMultihopQA ===
train rows: 2,500
val rows  : 125
eval rows : 500
train contexts per question: {5: 2500}

=== yixuantt/MultiHopRAG ===
train rows: 2,000
val rows  : 50
eval rows : 500
train contexts per question: {5: 2000}

All dataset fields are aligned.


In [4]:
for dataset_name, dataset in pipeline.dataset_constructor.datasets.items():
    sample = {
        "query": dataset.train_ds["query"][0],
        "answer": dataset.train_ds["answer"][0],
        "gold_context": dataset.train_ds["gold_context"][0],
        "distractor": dataset.train_ds["distractor"][0],
    }
    print(f"\n=== Sample: {dataset_name} ===")
    pprint(sample, width=120)


=== Sample: hotpotqa/hotpot_qa ===
{'answer': '12 member universities',
 'distractor': ['Connecticut posted a 20–3 record, earned the Yankee Conference championship with a 10–0 regular '
                'season to claim the automatic bid to the 1959 NCAA University Division Baseball Tournament.\n'
                '\n'
                ' They were an automatic selection to the 1959 College World Series for District 1, their second '
                'appearance in the ultimate college baseball event.\n'
                '\n'
                ' The Huskies lost their first game against <a href="">Penn State\n'
                '\n'
                'The Washington Huskies baseball team is the varsity intercollegiate baseball team of the University '
                'of Washington, located in Seattle, Washington, United States.\n'
                '\n'
                ' The program has been a member of the NCAA Division I Pac-12 Conference since the start of the 1960 '
                'season, 

## 3. Prepare and train the HyperNet models

Each method receives the shared datasets. Training and validation contexts/queries are embedded and tokenized before training.

### 3.1 Type_MeanEmbeddings

In [ ]:
mean_pipeline = pipeline.use_embd2adapter("mean_embds")
mean_pipeline.hypernet_trainer.prepare_datasets()

print(f"MeanEmbeddings train rows: {len(mean_pipeline.hypernet_trainer.train_ds):,}")
print(f"MeanEmbeddings val rows:   {len(mean_pipeline.hypernet_trainer.val_ds):,}")

Embedding training contexts:   0%|          | 0/391 [00:00<?, ?batch/s]

Saved training embeddings to C:\Atelier\Research\Embd2Adapter_for_RAG\Notebooks\outputs\cache\embeddings_cache.npy


Embedding training queries:   0%|          | 0/79 [00:00<?, ?batch/s]

Saved training query embeddings to C:\Atelier\Research\Embd2Adapter_for_RAG\Notebooks\outputs\cache\embeddings_cache_queries.npy


Parameter 'function'=<function HyperNetTrainer.tokenize.<locals>.tokenize_batch at 0x000001C1CE2F6770> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


...

Saved validation embeddings to C:\Atelier\Research\Embd2Adapter_for_RAG\Notebooks\outputs\cache\embeddings_cache_validation.npy


Embedding validation queries:   0%|          | 0/4 [00:00<?, ?batch/s]

Saved validation query embeddings to C:\Atelier\Research\Embd2Adapter_for_RAG\Notebooks\outputs\cache\embeddings_cache_validation_queries.npy


Tokenizing validation dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Validation tokenization truncation stats:
  total_rows: 450
  truncated_rows: 0
  truncated_ratio: 0.0
  max_raw_sequence_length: 1409
  total_removed_tokens: 0
  max_removed_tokens: 0
MeanEmbeddings train rows: 10,000
MeanEmbeddings val rows:   450


In [7]:
mean_log_history, mean_hypernet_spec = mean_pipeline.train()
mean_hypernet_spec

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Embedding model unloaded: bge-m3:latest


Step,Training Loss,Validation Loss,Token F1
150,0.761130,0.566151,0.686896
300,0.490967,0.476742,0.746319
450,0.423039,0.450815,0.730229
600,0.387654,0.435493,0.731165
750,0.380264,0.410171,0.754696
900,0.392948,0.402485,0.741175
1050,0.388938,0.408052,0.750475
1200,0.362452,0.400538,0.745091
1350,0.355147,0.388262,0.771663
1500,0.311540,0.384945,0.773636


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Parameters: 78,120,320
Trainable:  78,120,320
Size:       149.00 MiB
Training history appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_13.json


{'num_parameters': 78120320,
 'trainable_parameters': 78120320,
 'parameter_size_bytes': 156240640,
 'parameter_size_mib': 149.002685546875}

### 3.2 Type_QueryDiffPooling

Projected context embeddings are normalized before self-attention and reused as its residual, while the projected query is independently normalized to the same scale. Their difference is passed through `diff_projector`; mean and max pooling are then concatenated and projected into the Controller input. The entire HyperNet is trained end-to-end using only the answer-label causal-LM loss.

In [ ]:
query_diff_pipeline = pipeline.use_embd2adapter("query_diff_pooling")
query_diff_pipeline.hypernet_trainer.prepare_datasets()

print(f"QueryDiffPooling train rows: {len(query_diff_pipeline.hypernet_trainer.train_ds):,}")
print(f"QueryDiffPooling val rows:   {len(query_diff_pipeline.hypernet_trainer.val_ds):,}")

VectorStore Initialized
Embd2Adapter Pipline Initialized!
Experiment Setup Completed. check the datasets whether it is correct or not, and begin training your model.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

HyperNetQueryDiffPoolingTrainer Initialized!
Hypernet with QueryDiffPooling Loaded.
Loaded training embeddings from C:\Atelier\Research\Embd2Adapter_for_RAG\Notebooks\outputs\cache\embeddings_cache.npy
Loaded training query embeddings from C:\Atelier\Research\Embd2Adapter_for_RAG\Notebooks\outputs\cache\embeddings_cache_queries.npy


Parameter 'function'=<function HyperNetTrainer.tokenize.<locals>.tokenize_batch at 0x000001BF01E616F0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Tokenizing training dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Training tokenization truncation stats:
  total_rows: 10000
  truncated_rows: 0
  truncated_ratio: 0.0
  max_raw_sequence_length: 1776
  total_removed_tokens: 0
  max_removed_tokens: 0
Loaded validation embeddings from C:\Atelier\Research\Embd2Adapter_for_RAG\Notebooks\outputs\cache\embeddings_cache_validation.npy
Loaded validation query embeddings from C:\Atelier\Research\Embd2Adapter_for_RAG\Notebooks\outputs\cache\embeddings_cache_validation_queries.npy


Tokenizing validation dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Validation tokenization truncation stats:
  total_rows: 450
  truncated_rows: 0
  truncated_ratio: 0.0
  max_raw_sequence_length: 1409
  total_removed_tokens: 0
  max_removed_tokens: 0
QueryDiffPooling train rows: 10,000
QueryDiffPooling val rows:   450


In [7]:
query_diff_log_history, query_diff_hypernet_spec = query_diff_pipeline.train()
query_diff_hypernet_spec

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Embedding model unloaded: bge-m3:latest


Step,Training Loss,Validation Loss,Token F1
150,0.817466,0.547313,0.681503
300,0.461276,0.496363,0.719647
450,0.409614,0.462015,0.725789
600,0.372631,0.421815,0.740340
750,0.366640,0.402734,0.766768
900,0.375939,0.394810,0.759014
1050,0.362017,0.411927,0.767523
1200,0.345042,0.389576,0.765035
1350,0.338256,0.392265,0.765203
1500,0.290031,0.400833,0.776879


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Parameters: 85,474,176
Trainable:  85,474,176
Size:       163.03 MiB
Training history appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_15.json


{'num_parameters': 85474176,
 'trainable_parameters': 85474176,
 'parameter_size_bytes': 170948352,
 'parameter_size_mib': 163.029052734375}

## 4. Load and evaluate the trained models

Load the saved HyperNet parameters, then evaluate the complete evaluation split.

### 4.1 MeanEmbeddings

In [ ]:
mean_pipeline = pipeline.use_embd2adapter("mean_embds")
mean_checkpoint = mean_pipeline.load_trained_hypernet()
mean_results = mean_pipeline.experimentHyperNet()
mean_results

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

HyperNetMeanEmbdsTrainer Initialized!
Hypernet with MeanEmbds Loaded.
Loaded trained HyperNet from outputs\mean_embds\hypernet_state_dict.pt


Evaluating hotpotqa/hotpot_qa (RAG):   0%|          | 0/1000 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_13.json


Evaluating hotpotqa/hotpot_qa (No RAG):   0%|          | 0/1000 [00:00<?, ?question/s]

...

Evaluating framolfese/2WikiMultihopQA (RAG):   0%|          | 0/500 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_13.json


Evaluating framolfese/2WikiMultihopQA (No RAG):   0%|          | 0/500 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_13.json


{'hotpotqa': {'rag': {'exact_match': 0.538, 'token_f1': 0.620357775736808},
  'no_rag': {'exact_match': 0.568, 'token_f1': 0.6540885036875749}},
 'multihoprag': {'rag': {'exact_match': 0.832, 'token_f1': 0.8466},
  'no_rag': {'exact_match': 0.834, 'token_f1': 0.8473999999999999}},
 'musique': {'rag': {'exact_match': 0.402, 'token_f1': 0.43980687704108756},
  'no_rag': {'exact_match': 0.618, 'token_f1': 0.6601040793296831}},
 '2wikimultihop': {'rag': {'exact_match': 0.646,
   'token_f1': 0.6715839105339105},
  'no_rag': {'exact_match': 0.672, 'token_f1': 0.6979496392496393}}}

### 4.2 QueryDiffPooling

In [4]:
query_diff_pipeline = pipeline.use_embd2adapter("query_diff_pooling")
query_diff_checkpoint = query_diff_pipeline.load_trained_hypernet()
query_diff_results = pipeline.experimentHyperNet()
query_diff_results

VectorStore Initialized
Embd2Adapter Pipline Initialized!
Experiment Setup Completed. check the datasets whether it is correct or not, and begin training your model.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

HyperNetQueryDiffPoolingTrainer Initialized!
Hypernet with QueryDiffPooling Loaded.
Loaded trained HyperNet from outputs\query_diff_pooling\hypernet_state_dict.pt


Evaluating hotpotqa/hotpot_qa (RAG):   0%|          | 0/1000 [00:00<?, ?question/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


...

Evaluating framolfese/2WikiMultihopQA (RAG):   0%|          | 0/500 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_15.json


Evaluating framolfese/2WikiMultihopQA (No RAG):   0%|          | 0/500 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_15.json


{'hotpotqa': {'rag': {'exact_match': 0.548, 'token_f1': 0.625139297630474},
  'no_rag': {'exact_match': 0.585, 'token_f1': 0.6616658774031839}},
 'multihoprag': {'rag': {'exact_match': 0.888, 'token_f1': 0.8975333333333333},
  'no_rag': {'exact_match': 0.872, 'token_f1': 0.8815333333333333}},
 'musique': {'rag': {'exact_match': 0.408, 'token_f1': 0.4535345044429255},
  'no_rag': {'exact_match': 0.618, 'token_f1': 0.6413779337105963}},
 '2wikimultihop': {'rag': {'exact_match': 0.636,
   'token_f1': 0.662062481962482},
  'no_rag': {'exact_match': 0.648, 'token_f1': 0.6724529581529581}}}

### 4.3 BaseModel

In [4]:
# Release the active HyperNet before loading the BaseModel pipeline.
#query_diff_pipeline.release_method()
base_pipeline = pipeline.use_base_model()
base_results = base_pipeline.experimentBaseModel()
base_results

Evaluating hotpotqa/hotpot_qa (RAG):   0%|          | 0/1000 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_15.json


Evaluating hotpotqa/hotpot_qa (No RAG):   0%|          | 0/1000 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_15.json


Evaluating yixuantt/MultiHopRAG (RAG):   0%|          | 0/500 [00:00<?, ?question/s]

...

Evaluating framolfese/2WikiMultihopQA (RAG):   0%|          | 0/500 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_15.json


Evaluating framolfese/2WikiMultihopQA (No RAG):   0%|          | 0/500 [00:00<?, ?question/s]

Evaluation Process Initialized
Result appended to C:\Atelier\Research\Embd2Adapter_for_RAG\Experiment\Result\experiment_2026_08_15.json


{'hotpotqa': {'rag': {'exact_match': 0.448, 'token_f1': 0.3208403041318227},
  'no_rag': {'exact_match': 0.472, 'token_f1': 0.3449720178769288}},
 'multihoprag': {'rag': {'exact_match': 0.444, 'token_f1': 0.3521111111111111},
  'no_rag': {'exact_match': 0.454, 'token_f1': 0.3450779220779221}},
 'musique': {'rag': {'exact_match': 0.102, 'token_f1': 0.07544919242941546},
  'no_rag': {'exact_match': 0.162, 'token_f1': 0.10449382691190424}},
 '2wikimultihop': {'rag': {'exact_match': 0.228,
   'token_f1': 0.1766447002267772},
  'no_rag': {'exact_match': 0.244, 'token_f1': 0.1986058235355422}}}